# 🤖 Agents: Emergent Tool Use

DSPy's ReAct module lets agents interleave reasoning with tool calls.
But the real magic: DSPy can **optimize the agent's decision-making**, not just its words.

In this notebook we'll:
1. Explore the tools available to our agents
2. Run a simple calculator agent
3. Run a search & synthesis agent against real ticket data
4. **Optimize** agent behavior — watching tool-use patterns emerge

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.tools import TOOL_REGISTRY, calculate, search_tickets, get_ticket_stats
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

print("🔧 Available Tools:")
for name, fn in TOOL_REGISTRY.items():
    print(f"  • {name}: {fn.__doc__.strip().split(chr(10))[0]}")

In [ ]:
# Tools are just Python functions — try them!
print("calculate('(45 * 3) + 17'):", calculate("(45 * 3) + 17"))
print()
print("search_tickets('network'):")
print(search_tickets("network"))
print()
print("get_ticket_stats('priority'):")
print(get_ticket_stats("priority"))

## Task 16: Calculator Agent — The Simplest Agent

The calculator agent receives a math word problem and must decide **when** to call the `calculate` tool.
It can reason step-by-step, invoke the calculator, read the result, and continue reasoning.

In [ ]:
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy
MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())

btn = run_button("Run Calculator Agent")
out = widgets.Output()

def on_run_calc(b):
    with out:
        out.clear_output()
        print(f"⏳ Running Calculator Agent on {model_dd.value}...")
        result = run_baseline("calculator_agent", model_dd.value, max_eval=5)
        display_score("Calculator Agent", result.score)
        display_results_table(result.individual_scores)

btn.on_click(on_run_calc)
display(widgets.HBox([model_dd, btn]), out)

## Task 17: Search & Synthesize — Using Your Data

This agent searches your actual ticket database, retrieves relevant tickets,
and synthesizes an answer. It must decide **what** to search for and **how** to combine results.

In [ ]:
btn2 = run_button("Run Search Agent")
out2 = widgets.Output()

def on_run_search(b):
    with out2:
        out2.clear_output()
        print(f"⏳ Running Search Agent on {model_dd.value}...")
        print("   The agent will search your ticket database and synthesize answers...\n")
        result = run_baseline("search_agent", model_dd.value, max_eval=5)
        display_score("Search Agent", result.score)
        display_results_table(result.individual_scores)

btn2.on_click(on_run_search)
display(btn2, out2)

## Optimizing Agent Behavior

Now let's see what happens when we optimize.
DSPy doesn't just find better words — it finds better patterns for **WHEN** and **HOW** to use tools.

The optimizer observes which tool-call sequences lead to correct answers and reinforces those patterns.

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in list_by_tier(4)],
    description="Agent Task:")
opt_dd = optimizer_picker()
opt_btn = run_button("Optimize Agent")
opt_out = widgets.Output()

def on_optimize_agent(b):
    with opt_out:
        opt_out.clear_output()
        task = get_task(task_dd.value)
        print(f"⏳ Optimizing {task.name} on {model_dd.value}...")
        print(f"   Using {opt_dd.value} optimizer. This may take a minute...\n")

        result = run_optimization(task_dd.value, model_dd.value, opt_dd.value, max_eval=5)

        display_improvement(result.baseline_score, result.optimized_score)
        display_prompt_diff(result.prompt_before, result.prompt_after,
            title="Agent Prompt: Before vs. After Optimization")

        display_insight("Agentic Optimization",
            f"The optimizer improved the agent from {result.baseline_score:.0%} to "
            f"{result.optimized_score:.0%}. It didn't just change words — it changed "
            "how the agent thinks about when to call tools and in what order.")

opt_btn.on_click(on_optimize_agent)
display(widgets.VBox([widgets.HBox([task_dd, model_dd]), opt_dd, opt_btn]), opt_out)

---

## Summary

Agents are the frontier. DSPy optimizes not just the prompt but the **agent's decision-making strategy**:

- **Tools** are plain Python functions — nothing magical
- **ReAct** interleaves reasoning and tool calls in a loop
- **Optimization** discovers which tool-use patterns produce correct answers
- The same optimizer that tunes prompts can tune **agent behavior**

**Next: [The Full Picture →](07_the_full_picture.ipynb)** — Bringing evaluation, optimization, and agents together.